# Barrilito — spike v2 (tasa de cuenca)

**No es Hito 3.** Next.js sigue parado. Este notebook fija la **definición** del headline antes del Front.

- Lee **solo** CSVs en `data/` (marts). Cero `raw_*`. Cero secretos.
- Headline = `headline.rate_bbl_dia` = **bbl/día de cuenca** (`prod / días del mes`). Warehouse 2026-09-12: **590 393** bbl/día, `rate_method=calendar_days`.
- `productivity_bbl_dia` ≈ 260 bbl/pozo-día → KPI, **no** el contador.
- Completaciones / rigs / Brent / exportaciones / oleoductos / breakeven **no** están en estos CSV. No se inventan.

El notebook v1 (`barrilito_spike.ipynb`) interpoló el grano viejo. No copiarlo al Front.

Marca: **Barrilito**. Repo: `vaca-muerta-pulse`.


## 1. Tokens (igual que v1)

| Token | Hex | Uso |
| --- | --- | --- |
| `bg` | `#141210` | fondo |
| `surface` | `#1E1B18` | cards |
| `text` | `#F3EDE3` | títulos y números |
| `muted` | `#A89F91` | disclaimer, ejes, unidades |
| `accent` | `#E0A04A` | petróleo / headline |
| `accent2` | `#C45C26` | hover / segundo rango |
| `line` | `#6F9B8F` | gas / serie secundaria |
| `grid` | `#2A2724` | grilla |
| `empty` | `#7A6A5A` | empty state |

Copy: **último mes oficial**, no “últimos 30 días” como si Cap. IV fuera diario. Disclaimer MUST: *simulación a partir de datos mensuales oficiales*.


In [1]:
from pathlib import Path
from datetime import datetime, time
from zoneinfo import ZoneInfo

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

DATA = Path("data")
ART = ZoneInfo("America/Argentina/Buenos_Aires")
AS_OF = datetime(2026, 9, 11, 19, 0, tzinfo=ZoneInfo("UTC")).astimezone(ART)
M3_TO_BBL = 6.28981077  # mismo var dbt; Front no lo recalcula cuando el mart esté rebuilt

BARRILITO = {
    "bg": "#141210",
    "surface": "#1E1B18",
    "text": "#F3EDE3",
    "muted": "#A89F91",
    "accent": "#E0A04A",
    "accent2": "#C45C26",
    "line": "#6F9B8F",
    "grid": "#2A2724",
    "empty": "#7A6A5A",
}

plt.rcParams.update({
    "figure.facecolor": BARRILITO["bg"],
    "axes.facecolor": BARRILITO["surface"],
    "axes.edgecolor": BARRILITO["grid"],
    "axes.labelcolor": BARRILITO["muted"],
    "axes.titlecolor": BARRILITO["text"],
    "text.color": BARRILITO["text"],
    "xtick.color": BARRILITO["muted"],
    "ytick.color": BARRILITO["muted"],
    "grid.color": BARRILITO["grid"],
    "grid.linewidth": 0.6,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "legend.facecolor": BARRILITO["surface"],
    "legend.edgecolor": BARRILITO["grid"],
    "legend.labelcolor": BARRILITO["text"],
})

def fmt_miles(x, _pos=None):
    return f"{x:,.0f}".replace(",", " ")

def thousands_axis():
    return FuncFormatter(lambda x, p: fmt_miles(x))

headline = pd.read_csv(DATA / "headline.csv", parse_dates=["periodo"])
monthly = pd.read_csv(DATA / "monthly_pulse.csv", parse_dates=["periodo"])
companies = pd.read_csv(DATA / "company_latest.csv")
areas = pd.read_csv(DATA / "area_latest.csv")
top5 = pd.read_csv(DATA / "company_top5_month.csv", parse_dates=["periodo"])

row = headline.iloc[0]
rate_cuenca_bbl = float(row.rate_bbl_dia)
rate_cuenca_m3 = float(row.rate_m3_dia)
rate_prod_bbl = float(row.productivity_bbl_dia)
rate_prod_m3 = float(row.productivity_m3_dia)
check_cuenca = float(row.prod_pet_m3) * M3_TO_BBL / float(row.days_in_month)

print("as_of ART", AS_OF.isoformat())
print("periodo Cap. IV", row.periodo.date())
print(f"mart rate_bbl_dia={rate_cuenca_bbl:,.0f}  method={row.rate_method}")
print(f"cuenca     {rate_cuenca_bbl:,.0f} bbl/día  ({rate_cuenca_m3:,.0f} m³/día)")
print(f"productiv. {rate_prod_bbl:,.1f} bbl/pozo-día  (tef_sum={row.tef_sum:,.0f})")
print("disclaimer:", row.disclaimer)
assert row.rate_method == "calendar_days"
assert abs(rate_cuenca_bbl - check_cuenca) < 0.1
assert rate_cuenca_bbl > 100_000
assert rate_prod_bbl < 1_000


as_of ART 2026-09-11T16:00:00-03:00
periodo Cap. IV 2025-12-01
mart rate_bbl_dia=590,393  method=calendar_days
cuenca     590,393 bbl/día  (93,865 m³/día)
productiv. 259.8 bbl/pozo-día  (tef_sum=70,455)
disclaimer: simulación a partir de datos mensuales oficiales


## 2. Por qué cambió la definición

`tef` es tiempo efectivo **por pozo**. `sum(prod_pet_m3) / sum(tef)` es un promedio ponderado de **productividad** (barriles por pozo-día). El spike v1 lo puso en la portada y el reloj diario mostró ~173 bbl “hoy”.

La pregunta de producto es *a qué ritmo produce Vaca Muerta no convencional*. Eso es producción del mes **sobre el calendario**:

```text
rate_m3_dia     = sum(prod_pet_m3) / days_in_month
rate_bbl_dia    = rate_m3_dia × 6.28981077
```

Dic-2025 Pulse: ~2.91 millones de m³ / 31 ≈ **590 mil bbl/día**. Misma orden de magnitud que las cifras públicas (~640k en otro mes/recorte).

El Front, cuando exista el mart rebuilt, lee `fct_barrilito_rate.rate_bbl_dia` (cuenca) y `productivity_bbl_dia` (KPI). **No** multiplica el factor en el browser.


In [2]:
midnight = datetime.combine(AS_OF.date(), time.min, tzinfo=ART)
elapsed = min((AS_OF - midnight).total_seconds(), 86400.0)
barrels_today = rate_cuenca_bbl * elapsed / 86400.0
barrels_today_old = rate_prod_bbl * elapsed / 86400.0
pace_per_sec = rate_cuenca_bbl / 86400.0

print(f"reloj ART {midnight.strftime('%H:%M')} → {AS_OF.strftime('%H:%M')}")
print(f"v2 cuenca     {barrels_today:,.0f} bbl hoy   @ {rate_cuenca_bbl:,.0f} bbl/día  ({pace_per_sec:.3f} bbl/s)")
print(f"v1 (incorrecto) {barrels_today_old:,.0f} bbl hoy   @ {rate_prod_bbl:.1f} bbl/pozo-día")
print("MUST:", row.disclaimer)
print("mes fuente: último mes oficial Cap. IV, no una ventana rodante de 30 días")


reloj ART 00:00 → 16:00
v2 cuenca     393,595 bbl hoy   @ 590,393 bbl/día  (6.833 bbl/s)
v1 (incorrecto) 173 bbl hoy   @ 259.8 bbl/pozo-día
MUST: simulación a partir de datos mensuales oficiales
mes fuente: último mes oficial Cap. IV, no una ventana rodante de 30 días


## 3. Portada — contador de cuenca + KPIs

Tres KPIs: ritmo de **cuenca**, volumen DDJJ en m³, productividad (para no esconder el número viejo: ahora se llama como es).


In [3]:
fig = plt.figure(figsize=(11, 7.2), constrained_layout=True)
gs = fig.add_gridspec(3, 3, height_ratios=[1.4, 0.58, 1.55])

hero = fig.add_subplot(gs[0, :])
hero.set_axis_off()
hero.set_facecolor(BARRILITO["bg"])
hero.text(0.0, 0.92, "BARRILITO", fontsize=13, color=BARRILITO["accent"], fontweight="bold")
hero.text(0.0, 0.78, "Vaca Muerta · no convencional · último mes oficial", fontsize=11, color=BARRILITO["muted"])
hero.text(0.0, 0.42, f"{barrels_today:,.0f}".replace(",", " "), fontsize=48, color=BARRILITO["text"], fontweight="bold", va="center")
hero.text(0.58, 0.42, "bbl hoy", fontsize=16, color=BARRILITO["accent"], va="center")
hero.text(
    0.0, 0.08,
    f"Ritmo de cuenca {rate_cuenca_bbl:,.0f} bbl/día · Cap. IV {int(row.anio)}-{int(row.mes):02d} · calendar_days\n"
    f"{row.disclaimer}",
    fontsize=11, color=BARRILITO["muted"], va="bottom",
)

kpi_specs = [
    (fmt_miles(rate_cuenca_bbl), "bbl/día cuenca", "prod / días del mes"),
    (fmt_miles(row.prod_pet_m3), "m³ petróleo", "DDJJ dic-2025"),
    (f"{rate_prod_bbl:.0f}", "bbl/pozo-día", "productividad (no es Barrilito)"),
]
for i, (value, unit, caption) in enumerate(kpi_specs):
    ax = fig.add_subplot(gs[1, i])
    ax.set_axis_off()
    ax.set_facecolor(BARRILITO["surface"])
    ax.text(0.06, 0.70, value, fontsize=16, color=BARRILITO["text"], fontweight="bold", transform=ax.transAxes)
    ax.text(0.06, 0.42, unit, fontsize=10, color=BARRILITO["accent"], transform=ax.transAxes)
    ax.text(0.06, 0.16, caption, fontsize=8, color=BARRILITO["muted"], transform=ax.transAxes)

monthly = monthly.copy()
monthly["days"] = monthly["periodo"].dt.days_in_month
monthly["rate_bbl_cuenca"] = monthly["prod_pet_m3"] * M3_TO_BBL / monthly["days"]

ax = fig.add_subplot(gs[2, :])
ax.plot(monthly["periodo"], monthly["rate_bbl_cuenca"] / 1e3, color=BARRILITO["accent"], linewidth=2.4, marker="o", markersize=5)
ax.fill_between(monthly["periodo"], monthly["rate_bbl_cuenca"] / 1e3, color=BARRILITO["accent"], alpha=0.18)
ax.set_title("Tasa de cuenca mensual · recorte Pulse (miles de bbl/día)")
ax.set_ylabel("10³ bbl/día")
ax.grid(True, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(bottom=0)

fig.suptitle("Portada v2 — el disclaimer viaja pegado al número", color=BARRILITO["muted"], fontsize=10, x=0.01, ha="left")
plt.show()
print("dic-2025 cuenca kbbl/d:", round(float(monthly.iloc[-1].rate_bbl_cuenca) / 1e3, 1))


dic-2025 cuenca kbbl/d: 590.4


/tmp/ipykernel_16821/2820135717.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Ranking de empresas (último mes Cap. IV)

Volumen del mes en m³, no tasa. Grano `idempresa × periodo`, sparse. No es el headline.


In [4]:
top_n = companies.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 5.5))
bars = ax.barh(top_n["empresa"], top_n["prod_pet_m3"] / 1e3, color=BARRILITO["accent"])
bars[-1].set_color(BARRILITO["accent2"])
ax.set_title("Top 10 empresas · dic-2025 · petróleo (miles de m³)")
ax.set_xlabel("10³ m³")
ax.xaxis.set_major_formatter(thousands_axis())
ax.grid(True, axis="x")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
share_ypf = companies.iloc[0].prod_pet_m3 / companies.prod_pet_m3.sum()
print(f"{companies.iloc[0].empresa} share dic-2025: {100 * share_ypf:.1f}%  ·  n={len(companies)}")


YPF S.A. share dic-2025: 55.6%  ·  n=22


/tmp/ipykernel_16821/2432699724.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Vista de área (permiso / concesión)

Grano preferido: `idareapermisoconcesion`. No yacimiento.


In [5]:
top_a = areas.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.barh(top_a["areapermisoconcesion"], top_a["prod_pet_m3"] / 1e3, color=BARRILITO["line"])
ax.set_title("Top 10 áreas (permiso/concesión) · dic-2025 · petróleo (miles de m³)")
ax.set_xlabel("10³ m³")
ax.xaxis.set_major_formatter(thousands_axis())
ax.grid(True, axis="x")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()
print("áreas en el mes:", len(areas), " · top:", areas.iloc[0].areapermisoconcesion)


áreas en el mes: 83  · top: LOMA CAMPANA


/tmp/ipykernel_16821/3676468432.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Top 5 a lo largo de 2025


In [6]:
fig, ax = plt.subplots(figsize=(11, 5))
order = (
    top5[top5["periodo"] == top5["periodo"].max()]
    .sort_values("prod_pet_m3", ascending=False)["empresa"]
    .tolist()
)
palette = [BARRILITO["accent"], BARRILITO["accent2"], BARRILITO["line"], "#D9C3A0", "#8C7B6B"]
for i, name in enumerate(order):
    g = top5[top5["empresa"] == name].sort_values("periodo")
    ax.plot(g["periodo"], g["prod_pet_m3"] / 1e3, label=name, color=palette[i], linewidth=2.2)
ax.set_title("Top 5 de dic-2025 a lo largo de 2025 (miles de m³)")
ax.set_ylabel("10³ m³")
ax.legend(frameon=True, fontsize=8)
ax.grid(True, axis="y")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()


/tmp/ipykernel_16821/439476202.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Lo que este recorte **no** trae

Capítulo IV (estos CSV) es producción pozo-mes. Empty state honesto; cero curva en cero.


In [7]:
missing = [
    ("Completaciones / fracturas", "Adjunto IV existe en CKAN; no está en el job Meltano. Sin fct_completions."),
    ("Rigs activos ahora", "SESCO perforación es mensual (pozos en perforación), no NCS/IAPG live."),
    ("Brent", "Commodity. Tap de enriquecimiento + copy ‘no es SE’, o no se muestra."),
    ("Exportaciones / destinos", "Comercio exterior SESCO, no este anual."),
    ("Capacidad de oleoductos", "Seed citada Oldelval/Otasa, o GIS Res. 319/93. No es Cap. IV."),
    ("Breakeven USD/bbl", "No hay serie pública. Fuera del análisis."),
]
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.set_axis_off()
ax.set_facecolor(BARRILITO["surface"])
ax.text(0.04, 0.92, "Sin dato en este recorte", fontsize=14, color=BARRILITO["text"], fontweight="bold", transform=ax.transAxes)
for i, (title, body) in enumerate(missing):
    y = 0.78 - i * 0.12
    ax.text(0.04, y, title, fontsize=11, color=BARRILITO["accent"], fontweight="bold", transform=ax.transAxes, va="top")
    ax.text(0.34, y, body, fontsize=10, color=BARRILITO["empty"], transform=ax.transAxes, va="top")
plt.tight_layout()
plt.show()


/tmp/ipykernel_16821/1273683784.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Mapa → Hito 3 (cuando arranque Next)

| Spike v2 | Next / Tremor | Fuente |
| --- | --- | --- |
| `barriles_hoy` desde tasa **cuenca** | `Metric` + tick 1s | `fct_barrilito_rate.rate_bbl_dia` (ya cuenca) |
| Disclaimer | MUST visible | columna `disclaimer` |
| KPI productividad | `Card` chico, no el hero | `productivity_bbl_dia` |
| Área chart bbl/día cuenca | `AreaChart` | agg mensual `prod / days_in_month` |
| Barras empresas / áreas | `BarList` | marts P1, volumen m³ |
| Empty fracturas y resto | `Callout` | no mart |

Warehouse 2026-09-12: `rate_bbl_dia` = 590 393, `rate_method=calendar_days`. El Front no debe copiar v1.

**Fuera de este spike:** Next, Vercel, mapa GIS, taps hermanos.


## 9. Checklist

- [x] Headline = cuenca (`prod / days_in_month`), ~590k bbl/día en dic-2025.
- [x] Productividad visible como KPI, no como ritmo del contador.
- [x] Reloj diario ART + disclaimer MUST.
- [x] Copy: último mes oficial, no telemetría.
- [x] Rankings empresa / área.
- [x] Empty state de fuentes que no están en Cap. IV.
- [x] `dbt build --select fct_barrilito_rate` en BQ (2026-09-12): `calendar_days`, CSV alineado.
- [ ] Hito 3 Next. Este notebook no lo cierra.
